# Datenakquise — Swiss Super League 2024/25

Zentrales Notebook: Führt alle drei Datenquellen nacheinander aus.

| Notebook | Quelle | Output |
|---|---|---|
| `fetch_data.ipynb` | API-Football (RapidAPI) | `raw/footballapi/` |
| `fetch_sportmonks.ipynb` | Sportmonks Football API v3 | `raw/sportmonks/` |
| `fetch_fbref.ipynb` | FBref (lokale HTML-Datei) | `raw/fbref/` |

**Voraussetzungen:**
- `.env` im Projekt-Root mit `API_FOOTBALL_KEY`, `API_FOOTBALL_URL`, `API_SPORTMONKS_KEY`, `API_SPORTMONKS_URL`
- FBref-HTML-Datei in `data_acquisition/fbref_html/Swiss Super League Stats _ FBref.com.html`

> **Hinweis:** Die CSVs in `raw/` sind im Git-Repository eingecheckt.  
> Dieses Notebook muss nur neu ausgeführt werden, wenn Rohdaten aktualisiert werden sollen.

In [ ]:
import subprocess
from pathlib import Path

NOTEBOOKS = [
    ("fetch_data.ipynb",        "API-Football   → raw/footballapi/"),
    ("fetch_sportmonks.ipynb",   "Sportmonks     → raw/sportmonks/"),
    ("fetch_fbref.ipynb",        "FBref (lokal)  → raw/fbref/"),
]

results = []
for nb_file, label in NOTEBOOKS:
    print(f"\n{'─'*55}")
    print(f"▶  {label}")
    print(f"   Notebook: {nb_file}")
    print(f"{'─'*55}")

    proc = subprocess.run(
        [
            "jupyter", "nbconvert",
            "--to", "notebook",
            "--execute",
            "--inplace",
            "--ExecutePreprocessor.timeout=600",
            nb_file,
        ],
        capture_output=True,
        text=True,
    )
    ok = proc.returncode == 0
    results.append((nb_file, ok))
    if ok:
        print(f"   ✅ Erfolgreich")
    else:
        print(f"   ❌ Fehler — letzter Stderr-Abschnitt:")
        print(proc.stderr[-1500:])

print(f"\n{'='*55}")
print("Zusammenfassung:")
for nb_file, ok in results:
    icon = '✅' if ok else '❌'
    print(f"  {icon}  {nb_file}")
print(f"{'='*55}")

In [ ]:
# Alle erzeugten CSVs anzeigen
from pathlib import Path
import pandas as pd

RAW_ROOT = Path("raw")
print(f"Gespeicherte Rohdaten in data_acquisition/raw/:\n")
for subfolder in sorted(RAW_ROOT.iterdir()):
    if not subfolder.is_dir():
        continue
    csvs = sorted(subfolder.glob("*.csv"))
    if not csvs:
        continue
    print(f"  📁 {subfolder.name}/")
    for csv_path in csvs:
        df   = pd.read_csv(csv_path)
        size = csv_path.stat().st_size / 1024
        print(f"     {csv_path.name:<42} {len(df):>3} Zeilen × {len(df.columns):>2} Spalten  ({size:.1f} KB)")
    print()
print("🏁 Alle Datenquellen eingelesen.")